In [2]:
# single-cell analysis package
library(Seurat)

# data science packages
library(stringr)
library(dplyr) 
library(tidyverse)

In [3]:
# Create the output directory
outDir <- file.path("./../data/")
if(!dir.exists(outDir)) dir.create(outDir,recursive=TRUE)

In [4]:
data_dir = "./../data/st_data/"
dir_list = list.dirs(data_dir, recursive = FALSE)

stOb_lst <- list()
patIDs <- list()

for (d in dir_list){
    #print(d)
    pat <- sub(".*SP", "P", d)
    stOb <- Seurat::Load10X_Spatial(
                  # The directory contains the read count matrix H5 file and the image data in a subdirectory called `spatial`. 
                  data.dir = d, 
                  assay = "Spatial", # specify name of the initial assay
                  slice = pat, # specify name of the stored image
                  filter.matrix = TRUE, 
                  to.upper = FALSE
                )
    stOb$patient <- pat
    print(stOb)
    stOb_lst <- append(stOb_lst, stOb)
    patIDs <- append(patIDs, pat)
}

An object of class Seurat 
33538 features across 2940 samples within 1 assay 
Active assay: Spatial (33538 features, 0 variable features)
 1 layer present: counts
 1 spatial field of view present: P1
An object of class Seurat 
33538 features across 1935 samples within 1 assay 
Active assay: Spatial (33538 features, 0 variable features)
 1 layer present: counts
 1 spatial field of view present: P2
An object of class Seurat 
33538 features across 1504 samples within 1 assay 
Active assay: Spatial (33538 features, 0 variable features)
 1 layer present: counts
 1 spatial field of view present: P3
An object of class Seurat 
33538 features across 3007 samples within 1 assay 
Active assay: Spatial (33538 features, 0 variable features)
 1 layer present: counts
 1 spatial field of view present: P4
An object of class Seurat 
33538 features across 2124 samples within 1 assay 
Active assay: Spatial (33538 features, 0 variable features)
 1 layer present: counts
 1 spatial field of view present: P5


In [5]:
# merge all seurat object into one seurat object
merged_stOb<- merge(stOb_lst[[1]], y = stOb_lst[c(2:length(stOb_lst))],
                                   add.cell.ids = levels(factor(unlist(patIDs))),
                                   merge.data = TRUE,
                                   project = "HGSOC" )

merged_stOb@meta.data$chemotherapy_response <- merged_stOb@meta.data$patient
merged_stOb@meta.data <- merged_stOb@meta.data %>%
                            mutate(chemotherapy_response = recode(chemotherapy_response, P1 = 'poor', P2 = 'good', P3 = "good", 
                                                                  P4 = "partial", P5 = "good", P6 = "partial", P7 = "poor", P8 = "poor"))

print(merged_stOb)

An object of class Seurat 
33538 features across 20123 samples within 1 assay 
Active assay: Spatial (33538 features, 0 variable features)
 8 layers present: counts.1, counts.2, counts.3, counts.4, counts.5, counts.6, counts.7, counts.8
 8 spatial fields of view present: P1 P2 P3 P4 P5 P6 P7 P8


In [6]:
# save unprocessed object
# saveRDS(merged_stOb, file= file.path(outDir, 'unprocessed_STdata.rds'))